# Skill Memory vs. ER

Fresh five-seed comparison on Split CIFAR-100. This notebook contains no saved experiment outputs.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL='https://github.com/kobros-tech/ocl_survey.git'
REPO_REF='feature/skill-memory-rigor-fixes'
COLAB_ROOT=Path('/content/ocl_survey')
if 'google.colab' in sys.modules and not (COLAB_ROOT/'experiments/main.py').exists():
    subprocess.run(['git','clone','--branch',REPO_REF,REPO_URL,str(COLAB_ROOT)],check=True)
REPO_ROOT=COLAB_ROOT if (COLAB_ROOT/'experiments/main.py').exists() else Path.cwd().resolve()
if not (REPO_ROOT/'experiments/main.py').exists(): raise FileNotFoundError('ocl_survey checkout not found')
RESULTS_ROOT=Path(os.environ.get('OCL_RESULTS_ROOT',REPO_ROOT/'results'))
SKILL=RESULTS_ROOT/'skill_memory_split_cifar100_20_2000'
ER=RESULTS_ROOT/'er_split_cifar100_20_2000'
SEEDS=(0,1,2,3,4)
print('Repository:',REPO_ROOT)
print('Reference:',REPO_REF if 'google.colab' in sys.modules else 'local checkout')
print('Results:',RESULTS_ROOT)


In [ ]:
# Bootstrap the exact project dependencies into the active notebook interpreter.
# This makes a fresh Colab runtime reproducible instead of relying on whatever
# packages happen to be preinstalled. Local environments that already satisfy
# requirements.txt will be unchanged by pip.
requirements=REPO_ROOT/'requirements.txt'
subprocess.run([sys.executable,'-m','pip','install','-r',str(requirements)],cwd=REPO_ROOT,check=True)
import hydra
import avalanche
print('Python:',sys.version.split()[0])
print('Hydra:',hydra.__version__)
print('Avalanche:',avalanche.__version__)


In [ ]:
subprocess.run([sys.executable,'experiments/run_skill_memory_replicates.py'],cwd=REPO_ROOT,check=True)


In [ ]:
def require_complete(root):
    missing=[str(root/str(s)/'logs.json') for s in SEEDS if not (root/str(s)/'logs.json').exists()]
    if missing: raise FileNotFoundError('Incomplete result set; missing:\n'+'\n'.join(missing))
require_complete(SKILL); require_complete(ER)
sys.path.insert(0,str(REPO_ROOT))
from src.toolkit.process_results import extract_results
from src.toolkit.post_metrics import compute_average_forgetting, compute_mean_std_metric
metric='Top1_Acc_Stream/eval_phase/test_stream/Task000'
def summarize(root,label):
    frame=extract_results(str(root),verbose=False)['training']
    if metric not in frame: raise KeyError(f'{label} missing {metric}')
    acc_mean,acc_std=compute_mean_std_metric(frame,metric)
    forg=compute_average_forgetting(frame.copy(),20)
    forg_mean,forg_std=compute_mean_std_metric(forg,'Average_Forgetting')
    return {'strategy':label,'final_accuracy_mean':acc_mean,'final_accuracy_std':acc_std,'final_forgetting_mean':forg_mean,'final_forgetting_std':forg_std,'seeds':frame['seed'].nunique()}
import pandas as pd
summary=pd.DataFrame([summarize(SKILL,'Skill Memory'),summarize(ER,'ER')])
summary


In [ ]:
import matplotlib.pyplot as plt
ax=summary.plot(x='strategy',y='final_accuracy_mean',kind='bar',yerr='final_accuracy_std',legend=False)
ax.set_ylabel('Final average accuracy'); ax.set_xlabel(''); ax.set_title('Skill Memory vs. ER — Split CIFAR-100')
plt.tight_layout(); plt.show()


## Integrity notes

The main benchmark uses adaptive Skill Memory only. Forced REUSE/CLONE/SCRATCH runs are mechanism diagnostics. Both methods use the same five seeds and OCL Survey result-processing. Missing runs fail the notebook instead of being silently mixed. Skill Memory model-state storage and ER replay storage are different resource types and should be reported separately.
